This section refreshes the building blocks you will lean on constantly: how a script starts, the core data structures (used idiomatically), and reading and writing files with `pathlib`.

::: {.callout-tip}
## Why learn this?
Every analysis starts the same way: get the data in, organise it, and iterate over it. Comprehensions, `collections`, and `pathlib` are the everyday glue that turns a folder of raw output into something you can compute on.

- **Imagine you need** to run the *same* post-processing on 50 cases from the cluster — a script plus a loop does all 50 unattended.
- **Imagine you need** to find every `field_0001.dat` buried in a deep results tree — that's super easy in Python using `Path.rglob('*.dat')`.
:::

## Program entry points

A Python file can be **imported** as a module or **run** as a script. The `__name__` guard lets the same file do both: code under it runs only when the file is executed directly, not when it is imported.

In [1]:
def main():
    print("running as a script")

# When run as `python myfile.py`, __name__ == '__main__'.
# When imported, __name__ == 'myfile' and main() does not fire.
if __name__ == "__main__":
    main()

running as a script


For real scripts, parse arguments with the standard-library `argparse` instead of reading `sys.argv` by hand — you get `--help`, types, and validation for free.

In [2]:
import argparse

def build_parser():
    p = argparse.ArgumentParser(description="Scale a value.")
    p.add_argument("x", type=float, help="input value")
    p.add_argument("--factor", type=float, default=2.0)
    return p

# Parse an explicit list here (a notebook has no command line):
args = build_parser().parse_args(["21", "--factor", "2"])
print(args.x * args.factor)

42.0


Run a module (not a file path) with `python -m package.module` — the same mechanism behind `python -m pip` and `python -m pytest`.

**Imagine you need** to group ten thousand particles by which grid cell they fall in, or tally how many snapshots hit each flow regime. This is super easy in Python using `collections` (`defaultdict`, `Counter`).

## Data structures

### Comprehensions
Comprehensions build lists, dicts, and sets in one readable expression — prefer them to manual `append` loops.

In [3]:
squares = [n ** 2 for n in range(6)]
evens = [n for n in range(10) if n % 2 == 0]
lookup = {ch: i for i, ch in enumerate('abc')}
unique = {n % 3 for n in range(10)}
print(squares)
print(evens)
print(lookup)
print(unique)

[0, 1, 4, 9, 16, 25]
[0, 2, 4, 6, 8]
{'a': 0, 'b': 1, 'c': 2}
{0, 1, 2}


### The `collections` module
Specialised containers that make intent clear and code shorter.

In [4]:
from collections import namedtuple, defaultdict, Counter, deque

# namedtuple: a lightweight, immutable record with named fields
Point = namedtuple("Point", ["x", "y"])
p = Point(1.0, 2.0)
print("namedtuple:", p, p.x, p.y)

# defaultdict: missing keys get a default automatically
groups = defaultdict(list)
for word in ["fig", "flow", "turb", "flux"]:
    groups[word[0]].append(word)
print("defaultdict:", dict(groups))

# Counter: tally hashable items
print("Counter:", Counter("mississippi"))

# deque: fast appends/pops at BOTH ends (a ring buffer, a queue, ...)
d = deque([2, 3], maxlen=3)
d.appendleft(1); d.append(4)
print("deque (maxlen=3):", d)

namedtuple: Point(x=1.0, y=2.0) 1.0 2.0
defaultdict: {'f': ['fig', 'flow', 'flux'], 't': ['turb']}
Counter: Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})
deque (maxlen=3): deque([2, 3, 4], maxlen=3)


**Imagine you need** to load every `stats_*.csv` your solver scattered across a deep results tree. This is super easy in Python using `pathlib.Path` and `rglob`.

## File I/O and `pathlib`

Use `pathlib.Path` for filesystem paths — it is OS-independent and far cleaner than string concatenation or `os.path`. Always open files in a `with` block so they are closed even if an error occurs.

In [5]:
from pathlib import Path
import tempfile

workdir = Path(tempfile.mkdtemp())   # a scratch directory for this demo

# Build paths with the / operator, not string joins:
data_file = workdir / "results" / "run01.txt"
data_file.parent.mkdir(parents=True, exist_ok=True)

# Simple whole-file write/read:
data_file.write_text("Re = 5000\nCd = 0.42\n")
print(data_file.read_text())

# Line-by-line with a context manager (best for large files):
with open(data_file, "r") as fh:
    for line in fh:
        key, _, value = line.partition("=")
        if value:
            print(f"{key.strip()} -> {float(value)}")

Re = 5000
Cd = 0.42

Re -> 5000.0
Cd -> 0.42


In [6]:
# Useful Path attributes and globbing
print("name:", data_file.name)
print("suffix:", data_file.suffix)
print("parent:", data_file.parent)
print("exists:", data_file.exists())
txt_files = list(workdir.rglob("*.txt"))
print("found:", [f.name for f in txt_files])

name: run01.txt
suffix: .txt
parent: /tmp/tmpwsff0s28/results
exists: True
found: ['run01.txt']


## Self-tests

**1.** Using a **dict comprehension**, build `{n: n**3 for n in 1..5}` (cubes).

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
cubes = {n: n ** 3 for n in range(1, 6)}
print(cubes)

**2.** Given `words = ['fig','flow','turb','flux','film']`, use `Counter` to find how many start with each letter.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
from collections import Counter
words = ['fig', 'flow', 'turb', 'flux', 'film']
print(Counter(w[0] for w in words))

**3.** Write the numbers 0–4 to a file `nums.txt` (one per line) with `pathlib`, then read them back into a list of ints.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
from pathlib import Path
import tempfile
f = Path(tempfile.mkdtemp()) / "nums.txt"
f.write_text("\n".join(str(n) for n in range(5)))
nums = [int(line) for line in f.read_text().splitlines()]
print(nums)

## Turbulence in practice: reading a velocity signal

Put the pieces together — write a short synthetic velocity signal $u(t)$ to a file, read it back with `pathlib`, and compute its mean and rms fluctuation with comprehensions.

In [10]:
import numpy as np
from pathlib import Path
import tempfile

rng = np.random.default_rng(0)
t = np.linspace(0, 1, 200)
u = 10 + 0.5 * np.sin(2 * np.pi * 8 * t) + 0.2 * rng.standard_normal(t.size)

f = Path(tempfile.mkdtemp()) / 'signal.csv'
f.write_text('\n'.join(f'{ti:.4f},{ui:.4f}' for ti, ui in zip(t, u)))

rows = [line.split(',') for line in f.read_text().splitlines()]
u_read = [float(r[1]) for r in rows]
mean = sum(u_read) / len(u_read)
rms = (sum((ui - mean) ** 2 for ui in u_read) / len(u_read)) ** 0.5
print(f'<u> = {mean:.3f} m/s,  u_rms = {rms:.3f} m/s')

<u> = 10.003 m/s,  u_rms = 0.362 m/s


**Self-test.** Build a list of the fluctuations $u' = u - \langle u\rangle$ with a comprehension and check their mean is $\approx 0$.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
fluct = [ui - mean for ui in u_read]
print('mean of u\' =', sum(fluct) / len(fluct))